## Install library for xml files

- Instal spark-xml plug in maven -repository - spark-xml(com.databricks)

- Goto => Databricks account ==> Click cluster ==> Libraries

- Click Libraries ==> Click install now

- Click Maven ==> Search in coordinates, paste this line or just search with spark-xml then select latest version

- com.crealytics: spark-xml Then install ==> Then copy the library name and use it in notepad “com.databricks:spark-xml_2.13:0.14.0”

### Reading and writing xlsx files in pyspark
- We can use com.databricks:spark-xml_2.13:0.14.0 library for reading and writing xml files in pyspark
- While Reading or writing xml files, we need to use header option option("header",True)

##### Options
Read
- path: Location of XML files. Accepts standard Hadoop globbing expressions.
- rowTag: The row tag to treat as a row. For example, in this XML <books><book><book>...</books>, the value would be book. Default is ROW.
- rootTag: The root tag to treat as a root. For example, in this XML <books><book><book>...</books>, the value would be books. Default is ROWS.
- samplingRatio: Sampling ratio for inferring schema (0.0 ~ 1). Default is 1. Possible types are StructType, ArrayType, StringType, LongType, DoubleType, BooleanType, TimestampType - and NullType, unless you provide a schema.
- excludeAttribute: Whether to exclude attributes in elements. Default is false.
- nullValue: The value to treat as a null value. Default is "".
- mode: The mode for dealing with corrupt records. Default is PERMISSIVE.
####### PERMISSIVE:
When it encounters a corrupted record, sets all fields to null and puts the malformed string into a new field configured by columnNameOfCorruptRecord.
When it encounters a field of the wrong data type, sets the offending field to null.
####### DROPMALFORMED: ignores corrupted records.
####### FAILFAST: throws an exception when it detects corrupted records.

- inferSchema: if true, attempts to infer an appropriate type for each resulting DataFrame column, like a boolean, numeric or date type. If false, all resulting columns are of string type. Default is true.
- columnNameOfCorruptRecord: The name of new field where malformed strings are stored. Default is _corrupt_record.
- attributePrefix: The prefix for attributes so that to differentiate attributes and elements. This is the prefix for field names. Default is _.
- - - valueTag: The tag used for the value when there are attributes in an element that has no child elements. Default is _VALUE.
- - charset: Defaults to UTF-8 but can be set to other valid charset names.
- ignoreSurroundingSpaces: Whether or not whitespaces surrounding values should be skipped. Default is false.
- rowValidationXSDPath: Path to an XSD file that is used to validate the XML for each row. Rows that fail to validate are treated like parse errors as above. The XSD does not otherwise affect the schema provided or inferred. If the same local path is not already also visible on the executors in the cluster, then the XSD and any others it depends on should be added to the Spark executors with SparkContext.addFile. In this case, to use local XSD /foo/bar.xsd, call addFile("/foo/bar.xsd") and pass "bar.xsd" as rowValidationXSDPath.

In [0]:
from pyspark.sql.types import *
customSchema = StructType([ \
    StructField("COMM", StringType(), True), \
    StructField("DEPTNO", StringType(), True), \
    StructField("EMPNO", StringType(), True), \
    StructField("ENAME", StringType(), True), \
    StructField("HIREDATE", StringType(), True), \
    StructField("JOB", StringType(), True), \
    StructField("MGR", StringType(), True), \
    StructField("SAL", StringType(), True)])

In [0]:
customSchema

Out[32]: StructType(List(StructField(COMM,StringType,true),StructField(DEPTNO,StringType,true),StructField(EMPNO,StringType,true),StructField(ENAME,StringType,true),StructField(HIREDATE,StringType,true),StructField(JOB,StringType,true),StructField(MGR,StringType,true),StructField(SAL,StringType,true)))

In [0]:
!pip /usr/bin/spark-shell   --packages com.databricks:spark-xml_2.12:0.9.0
    # We can install pip also or via cluster also
    

#### Reading xml file

In [0]:
%fs ls /FileStore/tables/emp_xml.xml

path,name,size
dbfs:/FileStore/tables/emp_xml.xml,emp_xml.xml,3166


In [0]:
# To read XML file as plain text

df_txt=spark.read.text("/FileStore/tables/emp_xml.xml")


In [0]:
display(df_txt)

value
""
""
7369
SMITH
CLERK
7902
17-12-80
800
""
20


In [0]:
df=spark.read.format("com.databricks.spark.xml").option("rootTag", "emp").option("rowTag","row").load("/FileStore/tables/emp_xml.xml")


---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
<command-210797032279924> in <module>
----> 1 df=spark.read.format("com.databricks.spark.xml").option("rootTag", "emp").option("rowTag","row").load("/FileStore/tables/emp_xml.xml")

/databricks/spark/python/pyspark/sql/readwriter.py in load(self, path, format, schema, **options)
    202         self.options(**options)
    203         if isinstance(path, str):
--> 204             return self._df(self._jreader.load(path))
    205         elif path is not None:
    206             if type(path) != list:

/databricks/spark/python/lib/py4j-0.10.9-src.zip/py4j/java_gateway.py in __call__(self, *args)
   1302 
   1303         answer = self.gateway_client.send_command(command)
-> 1304         return_value = get_return_value(
   1305             answer, self.gateway_client, self.target_id, self.name)
   1306 

/databricks/spark/python/pyspark/sq

In [0]:
df.printSchema()

- Here schema are long etc, below used customschema and converted everything to stringtype

In [0]:
display(df)

### Reading data stating custom schema during load

In [0]:
emp_df = spark.read.format('xml').option("rootTag", "emp").option("rowTag","row").load('/FileStore/tables/emp_xml.xml',schema=customSchema)

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
<command-4262164550680202> in <module>
----> 1 emp_df = spark.read.format('xml').option("rootTag", "emp").option("rowTag","row").load('/FileStore/tables/emp_xml.xml',schema=customSchema)

/databricks/spark/python/pyspark/sql/readwriter.py in load(self, path, format, schema, **options)
    202         self.options(**options)
    203         if isinstance(path, str):
--> 204             return self._df(self._jreader.load(path))
    205         elif path is not None:
    206             if type(path) != list:

/databricks/spark/python/lib/py4j-0.10.9-src.zip/py4j/java_gateway.py in __call__(self, *args)
   1302 
   1303         answer = self.gateway_client.send_command(command)
-> 1304         return_value = get_return_value(
   1305             answer, self.gateway_client, self.target_id, self.name)
   1306 

/databricks/spark/python/pysp

In [0]:
display(emp_df)

In [0]:
display(df)

- Here schema are stringtype after using customschema , earlier it was long etc

### Writing (Saving data into XML) using xml or com.databricks.spark.xml with options rootTag and rowTag

In [0]:
emp_df.select("*").write.format('com.databricks.spark.xml').option("rootTag","emp").option("rowTag","row").save('/FileStore/tables/5xml',mode="overwrite")

In [0]:
%fs
ls /FileStore/tables/5xml/